# 06 Final Report
## Fraud Detection Decision Engine

This final report consolidates the complete Fraud Detection Decision Engine project, from business understanding and exploratory analysis to model development, model selection, and operational decision routing.

The project uses the IEEE-CIS Fraud Detection dataset to investigate how transaction-level fraud risk can be identified and translated into practical business actions.

Rather than treating fraud detection as a purely binary classification problem, this project approaches fraud as a risk management problem.

The machine learning model produces a fraud probability score, while the Decision Engine translates that score into operational actions based on the trade-off between:

- fraud detection,
- customer friction,
- and operational workload.

This report summarizes the key findings, evaluates the hypotheses developed throughout the analysis, consolidates the final modeling and Decision Engine results, and presents the main business implications, limitations, and recommendations.

## Executive Summary

This project develops an end-to-end Fraud Detection Decision Engine using the IEEE-CIS Fraud Detection dataset.

The objective is not only to classify transactions as fraudulent or legitimate, but to translate transaction-level fraud risk into operational actions that balance:

- fraud detection,
- customer friction,
- and manual review workload.

The dataset contains **590,540 transactions**, of which **20,663 transactions (3.50%)** are labeled as fraud.

Exploratory analysis showed that fraud risk is highly contextual. Fraud concentration varies across product categories, card characteristics, device information, and transaction behavior. These findings support the use of a multivariate machine learning approach rather than relying primarily on isolated fraud rules.

Four standalone models were evaluated:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

Additional ensemble combinations were also tested.

After evaluating Precision, Recall, F1-Score, ROC-AUC, Average Precision, threshold behavior, model complexity, and operational implications, **Random Forest was selected as the final fraud risk-scoring model**.

The final Decision Engine translates the Random Forest risk score into three actions:

| Risk Score | Action |
|---|---|
| `< 0.20` | Approve |
| `0.20 ≤ score < 0.50` | Additional Authentication |
| `≥ 0.50` | Manual Review |

On the held-out evaluation set:

- **96.94%** of transactions are routed to Approve.
- **1.55%** are routed to Additional Authentication.
- **1.51%** are routed to Manual Review.
- The two intervention bands together contain **65.40% of observed fraud cases while affecting only 3.06% of transactions**.
- The Manual Review band reaches an observed fraud rate of **94.40%**.

These results demonstrate that fraud intervention can be concentrated on a relatively small portion of transaction volume while preserving a low-friction path for most transactions.

A financial exposure analysis also identified an important limitation of probability-only routing.

Although the Approve band contains **34.60% of observed fraud cases**, those cases represent approximately **42.43% of observed fraud transaction amount**.

This indicates that **fraud likelihood and financial severity are not identical risk dimensions**. A future version of the Decision Engine should therefore evaluate transaction value as a secondary factor in addition to fraud probability.

The current framework should be interpreted as a **fraud-risk routing and decision-support system**, not as an estimate of fraud prevented or financial savings.

Before production deployment, the framework would require additional validation including:

- out-of-time testing,
- independent threshold selection,
- probability calibration assessment,
- operational capacity validation,
- intervention-outcome measurement,
- and a business cost framework.

Overall, the project demonstrates how machine learning outputs can be translated into practical business decisions rather than ending at model prediction alone.

## 1. Business Problem & Project Objective

Online payment fraud creates a business problem that extends beyond the direct value of fraudulent transactions.

An ineffective fraud detection system can create several forms of business impact:

- **Financial loss** from fraudulent transactions, chargebacks, and investigation costs.
- **Customer friction** when legitimate transactions are unnecessarily challenged or blocked.
- **Operational cost** when too many transactions require manual investigation.
- **Customer trust and reputational risk** when fraud controls are either too weak or excessively restrictive.

This creates an important trade-off.

A fraud system should detect as much fraudulent activity as possible, but maximizing fraud detection without considering false positives may create excessive customer friction and an unsustainable operational workload.

Therefore, the project is built around the following business question:

> **How can transaction-level fraud risk be identified and translated into practical operational actions while balancing fraud detection, customer friction, and manual review workload?**

The project has two connected objectives:

1. **Develop a machine learning risk-scoring model** capable of distinguishing higher-risk transactions from lower-risk transactions.
2. **Translate fraud probability scores into operational actions** through a Decision Engine rather than relying on a single binary fraud prediction.

The intended outcome is therefore not merely a fraud classifier, but a decision-support framework that helps determine how transactions with different levels of fraud risk should be handled.

## 2. Data & Fraud Landscape

The analysis is based on the IEEE-CIS Fraud Detection dataset, which contains real-world e-commerce transaction data together with transaction, payment, device, and identity-related attributes.

The primary training dataset contains:

| Metric | Value |
|---|---:|
| Total Transactions | 590,540 |
| Legitimate Transactions | 569,877 |
| Fraud Transactions | 20,663 |
| Fraud Rate | 3.50% |

Fraud therefore represents a relatively small proportion of the overall transaction population.

This class imbalance has an important implication for model evaluation.

A model could classify most transactions as legitimate and still achieve high overall accuracy while failing to detect a meaningful share of fraudulent activity.

For this reason, the project places greater emphasis on metrics that better reflect fraud detection performance, particularly:

- **Recall**, to measure how much observed fraud is successfully identified.
- **Precision**, to evaluate how concentrated fraud is among flagged transactions.
- **Average Precision (PR-AUC)**, to assess fraud-ranking quality under class imbalance.
- **ROC-AUC**, as a supporting measure of overall discrimination ability.

The dataset also contains a large and heterogeneous feature space covering transaction behavior, payment characteristics, anonymized engineered variables, and partially available identity information.

This makes fraud detection a multivariate behavioral problem rather than one that can be reliably solved using a single transaction attribute.

## 3. Key Exploratory Findings

The exploratory analysis showed that fraud is not distributed randomly across transactions. Several transaction, payment, temporal, and identity-related attributes consistently reveal different levels of fraud risk.

### 3.1 Product Context Is a Major Fraud Signal

`ProductCD` emerged as one of the strongest contextual variables in the exploratory analysis.

Product W represents approximately 74% of all transactions and therefore creates the largest overall business exposure. However, its fraud rate is relatively low at approximately 2.04%.

In contrast, Product C represents only approximately 11.6% of transactions but records a fraud rate of approximately 11.69%.

This distinction highlights two different dimensions of fraud risk:

- **High-volume exposure**, where a large transaction population can generate substantial total fraud even with a moderate fraud rate.
- **High-risk concentration**, where a smaller transaction segment contains a disproportionately large share of fraudulent activity.

### 3.2 Card Attributes Provide Important Contextual Signals

Several card-related attributes reveal distinct fraud patterns.

Credit-card transactions have a substantially higher fraud rate than debit-card transactions. Their overall fraud rate is approximately 6.68%, compared with 2.43% for debit transactions.

The difference becomes even stronger within certain ProductCD segments. For example, within Product C, credit-card transactions record a fraud rate of approximately 16.92%, compared with 8.16% for debit-card transactions.

Other card-related variables also identify distinct risk segments:

- `card3 = 185` shows a relatively high fraud concentration.
- `card5 = 137` represents a high-risk segment.
- `card5 = 226` represents a large-volume segment contributing substantial fraud exposure.

These findings suggest that card attributes are more informative when interpreted together with transaction context rather than as standalone fraud indicators.

### 3.3 Transaction Volume Alone Does Not Explain Fraud Risk

High transaction volume does not necessarily imply high fraud risk.

Several large-volume transaction groups maintain relatively low fraud rates, while some smaller groups exhibit substantially higher fraud concentrations.

The temporal analysis produced a similar finding. Changes in daily fraud rate were not directly aligned with changes in transaction volume, suggesting that changes in fraud activity cannot be explained by transaction volume alone.

Hourly transaction patterns also showed substantial similarity between legitimate and fraudulent transactions, indicating that hour-of-day by itself is unlikely to provide strong fraud discrimination.

### 3.4 Identity and Device Information Provide Complementary Signals

Identity-related attributes provide additional behavioral information, but their value appears strongest when combined with transaction features.

`DeviceType`, for example, shows meaningful differences in fraud rates:

- Mobile transactions: approximately 10.17%
- Desktop transactions: approximately 6.52%
- Transactions without DeviceType information: approximately 2.10%

Mobile transactions also show consistently higher fraud rates than desktop transactions across several ProductCD categories.

However, these patterns do not justify using DeviceType as a standalone fraud rule. Instead, identity and device information are treated as complementary signals within the predictive model.

### 3.5 Missingness Can Contain Behavioral Information

A substantial portion of the identity-related data is missing.

The exploratory analysis suggests that this missingness should not automatically be treated only as a data-quality problem.

For some attributes, missing information may reflect differences in how transactions are processed or whether identity information was collected.

This finding directly influenced the feature engineering strategy: missing values were handled differently depending on whether they represented ordinary incomplete observations or potentially meaningful behavioral absence.

## 4. Hypothesis Review

The exploratory analysis generated several hypotheses about potential drivers of fraud behavior.

The final report distinguishes between patterns that are directly supported by the analysis and explanations that remain unverified.

| Hypothesis | Evidence from the Project | Status |
|---|---|---|
| Temporal changes in fraud activity are not explained by transaction volume alone. | Fraud rate increased after the first month even though average transaction volume declined. Later changes in fraud rate also did not closely follow transaction volume. | **Supported by EDA** |
| Fraudsters may experience an adaptation phase before increasing attack intensity. | A shift from a lower fraud-rate period to a sustained higher fraud-rate period was observed. However, the dataset does not contain information about attacker identity, intent, or strategy. | **Not Tested / Speculative** |
| Temporal information may provide predictive value for fraud detection. | Fraud behavior changes over the observation period and `TransactionDT` was retained for modeling. However, no feature-ablation experiment was conducted to measure the incremental predictive contribution of temporal information. | **Partially Supported** |
| Hour-of-day alone is unlikely to be a strong standalone fraud predictor. | Legitimate and fraudulent transactions follow broadly similar hourly activity patterns, with fraud volume largely following overall transaction activity. | **Supported by EDA** |
| Product C contains characteristics associated with elevated fraud risk. | Product C represents a relatively small share of transactions but has a substantially higher fraud rate than Product W. However, the underlying causal reason for this concentration cannot be identified from the available data. | **Partially Supported** |
| The predictive value of card-related attributes depends on ProductCD. | Fraud rates for card network and card type vary considerably across ProductCD categories, indicating meaningful interaction between transaction product and payment characteristics. No dedicated interaction-ablation experiment was conducted during modeling. | **Partially Supported** |

### Interpretation

The hypothesis review reinforces an important distinction between **observed patterns** and **causal explanations**.

Several fraud patterns are clearly visible in the data, particularly differences across transaction products, card characteristics, and time periods.

However, the dataset does not provide sufficient evidence to determine why these patterns occur.

For example, the analysis can establish that Product C has a higher fraud concentration, but it cannot establish whether this results from product design, customer behavior, fraudster preference, authentication differences, or another operational factor.

Similarly, temporal changes in fraud activity are observable, but attributing them to fraudster adaptation would require additional longitudinal or behavioral evidence.

These unresolved questions should therefore be treated as areas for future investigation rather than conclusions of the current project.

## 5. Feature Engineering & Data Preparation

The feature engineering stage transformed the merged transaction and identity data into a complete and memory-efficient dataset suitable for model development.

### 5.1 Feature Structure

After removing `TransactionID`, which functions as a unique transaction identifier rather than a predictive behavioral feature, the dataset contained:

- **401 numerical features**
- **32 categorical features**
- **1 target variable (`isFraud`)**

The resulting feature space retained transaction, card, behavioral, anonymized V-series, device, and identity-related information identified during the exploratory analysis.

### 5.2 Missing Value Strategy

Missing values were not handled using a single universal rule.

The exploratory analysis indicated that missing identity information may itself contain behavioral information. Therefore, different feature groups were treated differently:

| Feature Group | Treatment |
|---|---|
| Transaction numerical features | Median imputation |
| Transaction categorical features | Explicit `"Missing"` category |
| Identity numerical features | Sentinel value `-999` |
| Identity categorical features | Explicit `"Missing"` category |
| Extremely sparse features | Evaluated according to their characteristics |

This approach preserves potentially informative identity missingness while applying conventional imputation to ordinary transaction features.

After preprocessing, no remaining missing values were present in the modeling dataset.

### 5.3 Computational Optimization

The complete feature-engineered dataset contained **590,540 rows and 434 columns**, including the target variable.

Because the dataset initially required approximately **2,060.63 MB** of memory, additional datatype optimization was performed.

Categorical string columns were converted to categorical datatypes and appropriate integer columns were downcast.

This reduced memory usage to approximately **922.55 MB**, representing a reduction of approximately **55.23%**, while preserving the full dataset.

The optimized dataset was then used as the input for model development.

## 6. Modeling Approach

The modeling stage evaluated multiple machine learning approaches to determine which model provides the most suitable fraud risk score for the Decision Engine.

### 6.1 Train-Test Split

The modeling dataset contains **433 predictor features** after separating the target variable.

A stratified train-test split was used with the following configuration:

- Training set: **472,432 transactions**
- Test set: **118,108 transactions**
- Test proportion: **20%**
- Random state: **42**
- Stratification based on `isFraud`

Stratification was used to preserve the fraud-class distribution across the training and testing datasets.

### 6.2 Categorical Encoding

The dataset contains **32 categorical features**.

These variables were transformed using `OrdinalEncoder`.

Unknown categories that may appear during inference are assigned the value `-1`, preventing the preprocessing pipeline from failing when previously unseen categories are encountered.

A consistent categorical encoding approach was retained across the candidate models to maintain the original feature dimensionality and avoid the substantial feature expansion that would result from one-hot encoding.

### 6.3 Feature Scaling

Feature scaling was applied only to Logistic Regression.

`StandardScaler` was fitted on the training data and used to standardize numerical features before Logistic Regression training.

The tree-based models were trained on the encoded but unscaled feature values because their split-based learning process does not require standardized numerical scales.

### 6.4 Candidate Models

Four standalone classification models were developed:

1. **Logistic Regression** — used as a baseline linear model.
2. **Random Forest** — a bagging-based tree ensemble.
3. **XGBoost** — a gradient-boosted tree model.
4. **LightGBM** — an efficient gradient-boosting implementation.

The objective was not simply to identify the model with the highest accuracy.

Because fraud represents only approximately 3.5% of the dataset, the models were evaluated using metrics that better reflect minority-class detection and ranking performance:

- Precision
- Recall
- F1-Score
- ROC-AUC
- Average Precision (PR-AUC)

### 6.5 Threshold Analysis

A default probability threshold of `0.50` was initially used to establish a consistent baseline across models.

However, a fixed threshold does not necessarily represent the most appropriate operating point for an imbalanced fraud detection problem.

Therefore, Precision, Recall, and F1-Score were also evaluated across multiple probability thresholds.

This threshold analysis was used to understand each model's Precision–Recall trade-off and support model comparison.

The resulting technical thresholds were treated as model evaluation operating points rather than final business decision boundaries.

Final operational thresholds were defined separately during the Decision Engine stage.

## 7. Model Performance & Final Model Selection

The four standalone models showed substantially different Precision–Recall behavior.

Because a fixed threshold of `0.50` favors some probability distributions more than others, the final technical comparison evaluated each candidate across the same threshold range and identified its strongest F1 operating point.

### 7.1 Threshold-Based Model Comparison

| Model | Best Threshold | Precision | Recall | F1-Score | ROC-AUC | Average Precision |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.85 | 47.27% | 39.34% | 42.94% | 0.8595 | 0.4178 |
| Random Forest | 0.20 | 75.46% | 64.87% | 69.76% | 0.9388 | 0.7365 |
| XGBoost | 0.25 | 77.60% | 56.59% | 65.45% | 0.9333 | 0.6909 |
| LightGBM | 0.85 | 71.12% | 54.34% | 61.61% | 0.9354 | 0.6468 |
| RF + XGBoost | 0.20 | 76.00% | 64.29% | 69.66% | 0.9460 | 0.7391 |
| RF + XGBoost + LightGBM | 0.40 | 73.69% | 62.74% | 67.77% | 0.9442 | 0.7158 |

Random Forest produced the strongest F1-Score among the evaluated candidates while maintaining the highest Recall among the strongest tree-based candidates.

The RF + XGBoost ensemble achieved slightly higher ROC-AUC and Average Precision, but the improvement was marginal and came with additional model complexity.

### 7.2 Business-Oriented Candidate Comparison

The final comparison focused particularly on Random Forest and the three-model ensemble because both offered strong fraud discrimination.

At their representative operating points:

| Model | Threshold | True Positives | False Negatives | False Positives | Flagged Transactions |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0.20 | 2,703 | 1,430 | 915 | 3,618 |
| RF + XGBoost + LightGBM | 0.40 | 2,593 | 1,540 | 926 | 3,519 |

Random Forest identified **110 additional fraud cases** while also producing **11 fewer false positives** than the three-model ensemble.

This result is particularly relevant because the project prioritizes reducing False Negatives while keeping unnecessary interventions at an acceptable level.

### 7.3 Final Model Selection

**Random Forest was selected as the primary risk-scoring model for the Fraud Detection Decision Engine.**

The decision was based on a combination of:

- strong fraud Recall,
- competitive Precision,
- the strongest overall Precision–Recall balance,
- strong Average Precision,
- lower implementation complexity than an ensemble,
- and favorable operational outcomes at the evaluated operating point.

The technical threshold of `0.20` used during model comparison was not treated as a complete business decision rule.

Instead, the selected Random Forest probability output was carried forward to the Decision Engine, where multiple risk bands were evaluated separately for operational routing.

## 8. Decision Engine Results

After Random Forest was selected as the primary risk-scoring model, the next stage translated its fraud risk score into operational actions.

A single binary threshold was considered insufficient because transactions with moderate risk should not necessarily receive the same treatment as transactions with very high fraud risk.

The final Decision Engine uses three operational risk bands:

| Risk Score | Action | Operational Purpose |
|---|---|---|
| `< 0.20` | **Approve** | Allow low-risk transactions to proceed without additional friction |
| `0.20 ≤ score < 0.50` | **Additional Authentication** | Apply a lower-cost verification step to medium-risk transactions |
| `≥ 0.50` | **Manual Review** | Reserve analyst investigation for the highest-risk transactions |

### 8.1 Final Routing Performance

The final Decision Engine produced the following routing distribution on the held-out test set:

| Action | Transactions | Fraud Cases | Transaction Share | Fraud Rate | Fraud Coverage |
|---|---:|---:|---:|---:|---:|
| Approve | 114,490 | 1,430 | 96.94% | 1.25% | 34.60% |
| Additional Authentication | 1,832 | 1,017 | 1.55% | 55.51% | 24.61% |
| Manual Review | 1,786 | 1,686 | 1.51% | 94.40% | 40.79% |

The two intervention bands together affect only approximately **3.06% of all transactions**, while containing approximately **65.40% of all observed fraud cases**.

### 8.2 Operational Interpretation

The routing structure creates a strong separation between transaction risk levels.

Approximately **96.94% of transactions** remain frictionless under the Approve action.

The Additional Authentication band represents only **1.55% of transaction volume**, but more than half of the transactions in this segment are fraudulent.

Manual Review is limited to approximately **1.51% of transactions**, while the fraud rate within this group reaches approximately **94.40%**.

This concentration is operationally important because manual investigation is assumed to be the most expensive intervention and should therefore be reserved for transactions with the strongest fraud risk signals.

The resulting framework reflects the project's main business priorities:

- preserve a low-friction experience for most transactions,
- use lower-cost intervention for medium-risk transactions,
- concentrate expensive analyst review on the highest-risk cases,
- and prioritize fraud coverage without attempting to maximize Recall at any operational cost.

### 8.3 Scope of the Decision Engine Evaluation

The Decision Engine evaluates **risk routing and operational workload**, not fraud-prevention effectiveness.

The dataset does not contain outcomes showing whether Additional Authentication or Manual Review would successfully stop a fraudulent transaction.

Therefore, fraud cases assigned to an intervention band should be interpreted as **fraud cases routed toward additional controls**, not as fraud cases prevented.

## 9. Financial Exposure by Decision Action

The original business objectives included understanding the financial impact associated with fraudulent transactions.

However, the available dataset does not contain actual chargeback losses, intervention costs, recovery amounts, or post-intervention outcomes.

Therefore, this section does **not** estimate:

- expected financial loss,
- fraud prevented,
- or financial savings from the Decision Engine.

Instead, the analysis evaluates the **observed transaction amount associated with confirmed fraud cases** across the three Decision Engine action bands.

This provides an additional financial exposure perspective while remaining grounded in outcomes directly available from the dataset.

In [9]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split


# Project paths
PROJECT_DIR = Path(
    "/home/rizalkurnia/Projects/DataAnalysis/fraud_detection"
)

DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
MODELS = PROJECT_DIR / "models"


# Load only the variables required for this analysis
evaluation_data = pd.read_parquet(
    DATA_PROCESSED / "data_optimized.parquet",
    columns=[
        "TransactionAmt",
        "isFraud"
    ]
)


# Reconstruct the same test split used during modeling
_, evaluation_test = train_test_split(
    evaluation_data,
    test_size=0.20,
    random_state=42,
    stratify=evaluation_data["isFraud"]
)

evaluation_test = evaluation_test.copy()


# Load Random Forest fraud probability
y_proba_rf = joblib.load(
    MODELS / "random_forest_probability.pkl"
)


# Basic validation
print(f"Test samples      : {len(evaluation_test):,}")
print(f"Probability count : {len(y_proba_rf):,}")
print(f"Fraud cases       : {evaluation_test['isFraud'].sum():,}")
print(f"Fraud rate        : {evaluation_test['isFraud'].mean():.2%}")

Test samples      : 118,108
Probability count : 118,108
Fraud cases       : 4,133
Fraud rate        : 3.50%


In [10]:
# Attach Random Forest probability to the evaluation test set
evaluation_test["fraud_probability"] = y_proba_rf


# Apply the final Decision Engine boundaries
evaluation_test["Action"] = pd.cut(
    evaluation_test["fraud_probability"],
    bins=[
        -float("inf"),
        0.20,
        0.50,
        float("inf")
    ],
    labels=[
        "Approve",
        "Additional Authentication",
        "Manual Review"
    ],
    right=False
)


# Total observed fraud transaction amount in the test set
total_fraud_amount = evaluation_test.loc[
    evaluation_test["isFraud"] == 1,
    "TransactionAmt"
].sum()


# Summarize financial exposure by action
financial_exposure = (
    evaluation_test
    .groupby("Action", observed=False)
    .agg(
        Transactions=("isFraud", "size"),
        Fraud_Cases=("isFraud", "sum"),
        Total_Transaction_Amount=("TransactionAmt", "sum"),
        Fraud_Transaction_Amount=(
            "TransactionAmt",
            lambda x: x[
                evaluation_test.loc[x.index, "isFraud"] == 1
            ].sum()
        )
    )
    .reset_index()
)


# Share of observed fraud transaction amount by action
financial_exposure["Fraud_Amount_Coverage"] = (
    financial_exposure["Fraud_Transaction_Amount"]
    / total_fraud_amount
)


financial_exposure

,Action,Transactions,Fraud_Cases,Total_Transaction_Amount,Fraud_Transaction_Amount,Fraud_Amount_Coverage
0,Approve,114490,1430,1.543961e+07,271201.437500,0.424326
1,Additional Authentication,1832,1017,2.889323e+05,176371.468750,0.275954
2,Manual Review,1786,1686,1.981892e+05,191561.765625,0.299721


In [11]:
# Fraud transactions only
fraud_only = evaluation_test[
    evaluation_test["isFraud"] == 1
].copy()


# Transaction amount profile for observed fraud cases
fraud_amount_profile = (
    fraud_only
    .groupby("Action", observed=False)
    .agg(
        Fraud_Cases=("TransactionAmt", "size"),
        Total_Fraud_Amount=("TransactionAmt", "sum"),
        Average_Fraud_Amount=("TransactionAmt", "mean"),
        Median_Fraud_Amount=("TransactionAmt", "median"),
        P90_Fraud_Amount=(
            "TransactionAmt",
            lambda x: x.quantile(0.90)
        )
    )
    .reset_index()
)


fraud_amount_profile.round(2)

,Action,Fraud_Cases,Total_Fraud_Amount,Average_Fraud_Amount,Median_Fraud_Amount,P90_Fraud_Amount
0,Approve,1430,271201.437500,189.649994,100.000000,420.25
1,Additional Authentication,1017,176371.484375,173.419998,77.000000,445.00
2,Manual Review,1686,191561.765625,113.620003,56.490002,280.00


### 9.1 Financial Exposure Interpretation

The financial exposure analysis reveals an important difference between **fraud likelihood** and **fraud transaction value**.

The Decision Engine successfully concentrates observed fraud cases into the intervention bands. Approximately 65.40% of observed fraud cases fall into Additional Authentication or Manual Review.

However, the distribution of fraud transaction amount is different.

| Action | Fraud Case Coverage | Fraud Amount Coverage |
|---|---:|---:|
| Approve | 34.60% | 42.43% |
| Additional Authentication | 24.61% | 27.60% |
| Manual Review | 40.79% | 29.97% |

Although the Manual Review band contains the largest number of observed fraud cases, those transactions are not the highest-value fraud cases on average.

Among observed fraud transactions:

- **Approve** has an average fraud transaction amount of approximately **189.65** and a median of **100.00**.
- **Additional Authentication** has an average of approximately **173.42** and a median of **77.00**.
- **Manual Review** has a lower average of approximately **113.62** and a median of **56.49**.

This indicates that the current Random Forest risk score is effective at identifying transactions with a high **likelihood of fraud**, but fraud probability does not necessarily correspond directly to the potential financial magnitude of a transaction.

The result is consistent with the current modeling objective: the model was trained to distinguish fraudulent from legitimate transactions, rather than to optimize directly for fraud transaction value or expected financial loss.

Therefore, the 42.43% fraud amount remaining in the Approve band should not be interpreted as an estimated realized loss. It represents the transaction value associated with observed fraud cases that received a low model risk score in the held-out evaluation data.

### 9.2 Future Amount-Aware Decision Strategy

A future version of the Decision Engine could incorporate transaction value as a **secondary decision dimension** in addition to the fraud risk score.

For example, selected high-value transactions with fraud scores below the current intervention threshold could receive an additional risk check rather than being routed solely according to probability.

Such a strategy may help address financially significant fraud cases that remain within the current Approve band.

However, no transaction-amount threshold is recommended in the current project because an amount-based intervention rule has not yet been evaluated against:

- additional customer friction,
- authentication volume,
- manual review workload,
- actual fraud losses,
- or intervention effectiveness.

Therefore, an amount-aware routing rule should be treated as a **future improvement requiring separate validation**, rather than as a modification to the current Decision Engine.

## 10. Business Insights

The project produces several business insights that extend beyond model performance metrics.

### 10.1 Fraud Risk Is Concentrated, but Business Exposure Has Multiple Dimensions

Fraud is not distributed uniformly across the transaction population.

The exploratory analysis identified segments with substantially different fraud characteristics. For example, Product C records a much higher fraud rate than Product W, while Product W remains important because its dominant transaction volume creates substantial total exposure.

This distinction shows that fraud management should consider at least two dimensions:

- **fraud likelihood**, and
- **transaction exposure or business volume**.

A segment with the highest fraud rate is not necessarily the segment responsible for the greatest overall financial exposure.

### 10.2 Transaction Context Is More Informative Than Isolated Rules

Several high-risk patterns emerge from combinations of transaction characteristics rather than from individual attributes alone.

For example, credit transactions have a higher overall fraud rate than debit transactions, and this difference becomes considerably stronger within Product C.

Similarly, card-related and device-related attributes show different fraud behavior depending on transaction context.

This supports the use of a multivariate machine learning model rather than relying primarily on simple standalone fraud rules.

### 10.3 Risk-Based Routing Can Concentrate Intervention on a Small Share of Transactions

The final Decision Engine demonstrates that operational intervention does not need to be applied broadly across the entire transaction population.

The selected routing strategy sends approximately **3.06% of transactions** toward Additional Authentication or Manual Review while those bands contain approximately **65.40% of observed fraud cases**.

At the same time, approximately **96.94% of transactions** remain in the low-friction Approve path.

This illustrates the practical value of using model risk scores to prioritize intervention rather than treating every transaction equally.

### 10.4 Manual Review Capacity Can Be Focused on Highly Concentrated Risk

Manual Review is the most operationally expensive action in the proposed framework.

The final configuration routes only approximately **1.51% of transactions** to Manual Review, while the observed fraud rate within this band reaches approximately **94.40%**.

This suggests that model-based prioritization can help focus limited analyst capacity on transactions with the strongest fraud signals rather than creating a broad manual-review queue.

The operational feasibility of this workload would still need to be validated against real transaction throughput and available analyst capacity.

### 10.5 Residual Risk Is an Intentional Trade-off

The Decision Engine does not attempt to intervene on every transaction that may eventually be identified as fraudulent.

Approximately **34.60% of observed fraud cases** remain within the Approve band.

This residual risk reflects the project's central trade-off: increasing fraud coverage further would require intervention on a larger number of legitimate transactions, increasing customer friction and operational cost.

The objective is therefore not to eliminate all fraud, but to allocate intervention where the model indicates that additional controls are most justified.

### 10.6 Fraud Probability and Financial Severity Are Not the Same

The financial exposure analysis introduced an additional insight.

Although the Approve band contains **34.60% of observed fraud cases**, those cases represent approximately **42.43% of the observed fraud transaction amount**.

Meanwhile, Manual Review contains **40.79% of fraud cases** but approximately **29.97% of fraud transaction amount**.

Fraud transactions remaining in the Approve band also show a higher average and median transaction value than fraud transactions routed to Manual Review.

This indicates that a transaction with a higher estimated fraud likelihood is not necessarily the transaction associated with the highest financial value.

The current Decision Engine therefore performs **fraud-risk prioritization**, not financial-loss optimization.

This distinction creates a clear opportunity for future development of an amount-aware risk strategy.

## 11. Business Recommendations

Based on the combined exploratory, modeling, Decision Engine, and financial exposure results, the following recommendations are proposed.

### 11.1 Use the Random Forest Decision Engine as the Primary Risk-Routing Framework

Random Forest should be retained as the primary fraud risk-scoring model for the current Decision Engine.

The final routing configuration is:

| Risk Score | Recommended Action |
|---|---|
| `< 0.20` | Approve |
| `0.20 ≤ score < 0.50` | Additional Authentication |
| `≥ 0.50` | Manual Review |

This structure provides differentiated treatment according to transaction risk rather than relying on a single binary fraud threshold.

On the held-out evaluation set, the two intervention bands contain approximately **65.40% of observed fraud cases while affecting only 3.06% of transactions**.

### 11.2 Reserve Manual Review for the Highest-Risk Transactions

Manual Review should remain concentrated on the highest-risk transactions.

The current `≥ 0.50` band represents approximately **1.51% of transactions** and has an observed fraud rate of approximately **94.40%**.

This makes the band suitable for prioritizing limited analyst capacity.

However, the 1.51% workload should not be assumed to be operationally feasible in production until it is compared with:

- actual daily transaction volume,
- available analyst capacity,
- average review time,
- and service-level requirements.

### 11.3 Use Additional Authentication as the Primary Lower-Cost Intervention

Transactions with fraud scores between `0.20` and `0.50` should receive an additional verification step rather than immediate manual investigation.

This allows the business to apply additional friction selectively while avoiding the operational cost of manually reviewing every medium-risk transaction.

The project does not contain intervention-outcome data, so the effectiveness of authentication methods cannot be estimated from the current dataset.

The Additional Authentication band should therefore be validated operationally before assuming that it reduces fraud losses.

### 11.4 Monitor High-Risk and High-Exposure Transaction Segments

The exploratory analysis identified several transaction segments that deserve continued monitoring.

Examples include:

- Product C, particularly credit transactions, because of its elevated fraud concentration.
- `card3 = 185` and `card5 = 137`, which show relatively high fraud rates.
- Product W and `card5 = 226`, which are important because of their large transaction volume and resulting exposure.

These segments should primarily be used for:

- monitoring,
- fraud reporting,
- targeted investigation,
- and model-performance diagnostics.

They should not automatically become hard transaction-blocking rules because the predictive model already incorporates these characteristics and standalone rules may create unnecessary overlap or customer friction.

### 11.5 Add Financial Severity as a Secondary Risk Dimension in a Future Version

The current Decision Engine prioritizes **fraud likelihood**, but the financial exposure analysis shows that fraud likelihood and transaction value are not always aligned.

Approximately **42.43% of observed fraud transaction amount** remains within the current Approve band, even though this band contains 34.60% of observed fraud cases.

A future Decision Engine should therefore evaluate whether selected high-value transactions require additional controls even when their fraud probability remains below the current intervention threshold.

This should be implemented only after testing alternative amount-aware rules against:

- fraud amount coverage,
- additional authentication volume,
- manual review workload,
- and legitimate customer friction.

No transaction-amount threshold is recommended by the current project because such a threshold has not yet been validated.

### 11.6 Monitor Model and Threshold Performance Over Time

Fraud behavior changes over time, and the exploratory analysis identified meaningful variation in fraud rates across the observation period.

Therefore, both model performance and Decision Engine routing should be monitored after deployment.

Important monitoring indicators include:

- fraud Recall,
- Precision,
- Average Precision,
- transaction share by action,
- fraud rate within each action band,
- fraud amount coverage,
- and changes in key transaction segments.

Material deterioration in these indicators should trigger investigation, threshold reassessment, or model retraining.

### 11.7 Validate the Framework Prospectively Before Production Deployment

The reported results should be treated as evidence from the current held-out evaluation rather than guaranteed production performance.

Before production use, the framework should be validated using a more realistic out-of-time evaluation and a separate dataset for threshold selection.

This would provide stronger evidence that the selected risk bands remain effective on future transactions rather than only on the historical sample used during project development.

## 12. Limitations

The project demonstrates a workable fraud risk-scoring and decision-routing framework, but several limitations should be considered when interpreting the results.

### 12.1 Random Split Does Not Simulate Future Fraud Behavior

The modeling stage uses a stratified random 80/20 train-test split.

While stratification preserves the fraud-class distribution, a random split does not fully represent a production environment where models are trained on historical transactions and applied to future transactions.

This is particularly relevant because the exploratory analysis identified changes in fraud behavior across time.

Therefore, the reported performance may not fully represent model performance under temporal distribution shift.

A chronological or out-of-time validation would provide a stronger production-oriented evaluation.

### 12.2 Threshold Selection Uses the Held-Out Test Data

Probability thresholds were explored using the same held-out test labels used to report model and Decision Engine performance.

This was useful for portfolio-level diagnostic comparison and understanding the Precision–Recall trade-off.

However, selecting or refining thresholds after observing test-set outcomes can introduce optimistic bias.

A stronger experimental design would separate:

- model training data,
- threshold-selection or validation data,
- and a final untouched test set.

Therefore, the current thresholds should be interpreted as project decision boundaries that require prospective validation rather than guaranteed production-optimal thresholds.

### 12.3 Intervention Effectiveness Cannot Be Estimated

The dataset contains fraud labels but does not contain outcomes from operational interventions.

It is therefore unknown whether:

- Additional Authentication would successfully stop a fraudulent transaction,
- a legitimate customer would successfully complete the verification process,
- or Manual Review would result in approval, rejection, or further investigation.

As a result, fraud cases routed to intervention bands represent **fraud coverage**, not fraud prevented.

### 12.4 Actual Financial Loss and Savings Are Not Available

`TransactionAmt` provides transaction value but does not represent realized financial loss.

The dataset does not include information such as:

- chargeback losses,
- recovered funds,
- investigation costs,
- authentication costs,
- customer-abandonment costs,
- or intervention effectiveness.

Therefore, the financial analysis measures **observed fraud transaction amount exposure**, not expected financial loss or financial savings generated by the Decision Engine.

### 12.5 Model Probability Has Not Been Formally Calibrated

The Random Forest probability output is used as a transaction-level fraud risk score.

However, no dedicated probability-calibration analysis was conducted.

Therefore, a score such as `0.70` should not automatically be interpreted as meaning that a transaction has an exactly 70% real-world probability of fraud.

The model scores are primarily used for **risk ranking and operational segmentation** in the current project.

### 12.6 Operational Capacity Is Based on Assumptions

The final Decision Engine routes approximately 1.51% of transactions to Manual Review.

This appears operationally concentrated within the evaluation dataset, but the project does not contain actual information about:

- production transaction throughput,
- analyst staffing,
- average investigation time,
- queue capacity,
- or required response times.

The practical feasibility of this workload therefore needs to be validated within a real operating environment.

### 12.7 Dataset Interpretability Is Limited

The IEEE-CIS dataset contains many anonymized or obfuscated variables, and identity information is only available for a subset of transactions.

This limits the ability to provide causal explanations for some observed fraud patterns.

For example, the project can identify elevated fraud concentration within certain ProductCD or card-related segments, but it cannot determine the underlying operational reason for those patterns.

### 12.8 Model Stability and Feature Contribution Require Further Testing

The current project does not include:

- out-of-time drift testing,
- probability calibration,
- formal feature-ablation experiments,
- or dedicated testing of hypothesized feature interactions.

Therefore, several exploratory relationships should be interpreted as descriptive evidence rather than proven causal or independently predictive relationships.

These limitations do not invalidate the current Decision Engine, but they define the boundary between what the project demonstrates and what would still need to be validated before production deployment.

## 13. Future Improvements

The current project establishes a complete fraud risk-scoring and decision-routing framework, but several improvements could strengthen both its predictive reliability and operational relevance.

### 13.1 Introduce Out-of-Time Validation

Future model evaluation should use a chronological validation design.

Transactions from earlier periods should be used for training, while later transactions should be reserved for validation and final testing.

This would better simulate the production setting in which a fraud model is always applied to transactions occurring after the data used for training.

Given the temporal variation observed during exploratory analysis, out-of-time validation would also help determine how sensitive model performance is to changes in fraud behavior.

### 13.2 Separate Model Evaluation and Threshold Selection

A stronger experimental design should use separate datasets for:

1. model training,
2. model and threshold validation,
3. and final untouched evaluation.

The validation set could be used to select operational boundaries, while the final test set would provide an unbiased estimate of the resulting Decision Engine performance.

This would reduce the optimistic bias that may result from evaluating thresholds on the same held-out data used during threshold exploration.

### 13.3 Evaluate Probability Calibration

The current Random Forest output functions effectively as a relative fraud risk score, but its probability calibration has not been formally evaluated.

Future work could compare calibration techniques such as:

- Platt scaling,
- isotonic regression,
- or other calibration methods.

Better-calibrated probabilities would become particularly valuable if operational decisions eventually depend on the absolute interpretation of fraud probability rather than only relative risk ranking.

### 13.4 Develop an Amount-Aware Decision Engine

The financial exposure analysis shows that fraud likelihood and transaction value are not perfectly aligned.

A future Decision Engine could therefore evaluate both:

- model fraud probability,
- and transaction financial exposure.

For example, high-value transactions with moderate or even relatively low fraud scores could receive an additional verification step.

Candidate amount-aware strategies should be tested systematically rather than introducing an arbitrary transaction-value threshold.

The evaluation should compare their impact on:

- fraud case coverage,
- fraud amount coverage,
- customer intervention volume,
- and manual review workload.

### 13.5 Introduce a Business Cost Framework

The original project objective considered financial loss, but the available dataset does not provide the information required to estimate it directly.

A production-oriented extension should incorporate actual business inputs such as:

- average fraud loss,
- chargeback cost,
- authentication cost,
- manual investigation cost,
- false-positive customer cost,
- and fraud recovery rate.

These inputs would allow threshold selection to move from metric-based optimization toward explicit **expected business cost or expected financial loss optimization**.

### 13.6 Measure Intervention Effectiveness

The effectiveness of Additional Authentication and Manual Review should be evaluated using real operational outcomes.

Future data collection should track whether:

- authentication was passed or failed,
- customers abandoned transactions,
- analysts approved or rejected transactions,
- confirmed fraud was prevented,
- and financial losses were ultimately realized.

This would allow the Decision Engine to be evaluated as an actual fraud-control system rather than only as a risk-routing framework.

### 13.7 Monitor Fraud and Model Drift

A deployed system should continuously monitor whether the relationship between model scores and fraud outcomes changes over time.

Monitoring should include:

- fraud rate,
- Recall and Precision,
- Average Precision,
- score distribution,
- action-band volumes,
- fraud concentration within each band,
- fraud amount coverage,
- and important transaction segments.

Significant deterioration should trigger investigation, threshold review, or model retraining.

### 13.8 Perform Feature Ablation and Interaction Analysis

Several exploratory findings suggest potentially meaningful interactions between transaction context, card characteristics, temporal information, device attributes, and missingness patterns.

Future experiments could remove or group feature families and measure the resulting change in model performance.

This would help distinguish features that are merely correlated with fraud from those that provide meaningful incremental predictive information.

It would also provide stronger evidence for hypotheses that were only descriptively supported during the current analysis.

### 13.9 Validate Operational Capacity Before Deployment

Finally, the proposed routing percentages should be translated into absolute operational volumes.

For example, a 1.51% Manual Review rate may represent a manageable queue in one business but an unsustainable workload in another.

Future deployment planning should therefore combine model outputs with:

- expected transaction throughput,
- analyst staffing,
- average case-handling time,
- queue limits,
- and response-time requirements.

Operational thresholds should ultimately reflect both fraud risk and the organization's actual capacity to intervene.

## 14. Final Conclusion

This project demonstrates that fraud detection is more useful when treated as a **risk management and operational decision problem** rather than only as a binary classification task.

The analysis began with a highly imbalanced dataset in which fraud represents approximately **3.50% of transactions**. Exploratory analysis showed that fraud risk varies substantially across transaction context, product categories, card characteristics, device information, and time, supporting the use of a multivariate modeling approach rather than isolated transaction rules.

Four standalone models were evaluated:

- Logistic Regression,
- Random Forest,
- XGBoost,
- and LightGBM.

Additional ensemble combinations were also examined.

After comparing Precision, Recall, F1-Score, ROC-AUC, Average Precision, threshold behavior, and operational implications, **Random Forest was selected as the final fraud risk-scoring model**.

The selected model was then integrated into a three-level Decision Engine:

| Fraud Risk Score | Operational Action |
|---|---|
| `< 0.20` | Approve |
| `0.20 ≤ score < 0.50` | Additional Authentication |
| `≥ 0.50` | Manual Review |

On the held-out evaluation set, this framework routes approximately **96.94% of transactions** through the low-friction Approve path.

Only approximately **3.06% of transactions** receive additional intervention, while these intervention bands contain approximately **65.40% of all observed fraud cases**.

Manual Review is further concentrated on only **1.51% of transactions**, where the observed fraud rate reaches approximately **94.40%**.

These results illustrate how machine learning risk scores can be translated into differentiated operational actions while balancing fraud detection, customer friction, and manual review workload.

The additional financial exposure analysis also reveals an important limitation of probability-only routing.

Although only **34.60% of observed fraud cases** remain in the Approve band, those transactions represent approximately **42.43% of observed fraud transaction amount**.

This shows that **fraud likelihood and financial severity are related but distinct dimensions of fraud risk**.

The current Decision Engine should therefore be understood as a **fraud-risk prioritization framework**, not a complete financial-loss optimization system.

Before production deployment, the framework would require stronger temporal validation, independent threshold selection, probability calibration assessment, real intervention outcomes, operational capacity testing, and a business cost framework.

Nevertheless, the project provides a complete end-to-end demonstration of how exploratory analysis, machine learning, threshold evaluation, and business rules can be connected to transform fraud prediction into a practical decision-support system.